<a href="https://colab.research.google.com/github/RenteriaRaul/Doctorado-DCAG/blob/main/notebooks/sustax/03_sustax_event_analysis_2015.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Configuración e imports**

In [1]:
from google.colab import drive
import os
import glob
import pandas as pd
import numpy as np

drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Doctorado /Probabilidad/Precipitación"
SUSTAX_DIR = os.path.join(PROJECT_DIR, "Datos Sustax")

OBS_TOTAL_FOLDER = os.path.join(SUSTAX_DIR, "OBS_COMPOSITES_TOTAL")
BYSCEN_TOTAL_FOLDER = os.path.join(SUSTAX_DIR, "SUSTAX_TOTAL_MERGED", "BY_SCENARIO_TOTAL")

OUT_FOLDER = os.path.join(SUSTAX_DIR, "EVENT_2015_ANALYSIS")
os.makedirs(OUT_FOLDER, exist_ok=True)

TARGET_DATE = "2015-10-23"
TARGET_MONTH = 10

print("PROJECT_DIR:", PROJECT_DIR)
print("OUT_FOLDER:", OUT_FOLDER)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/Doctorado /Probabilidad/Precipitación
OUT_FOLDER: /content/drive/MyDrive/Doctorado /Probabilidad/Precipitación/Datos Sustax/EVENT_2015_ANALYSIS


# **Helpers y carga de OBS**

In [2]:
def normalize_date_series_to_iso(s):
    d0 = pd.to_datetime(s, errors="coerce", dayfirst=False)
    d1 = pd.to_datetime(s, errors="coerce", dayfirst=True)
    dt = d1 if d1.notna().sum() > d0.notna().sum() else d0
    dt = dt.dt.normalize()
    return dt.dt.strftime("%Y-%m-%d")

def normalize_date_series_to_dt(s):
    d0 = pd.to_datetime(s, errors="coerce", dayfirst=False)
    d1 = pd.to_datetime(s, errors="coerce", dayfirst=True)
    dt = d1 if d1.notna().sum() > d0.notna().sum() else d0
    return dt.dt.normalize()

def pick_precip_col(df):
    meta_cols = {"sustax_total", "lat", "lon", "date"}
    candidates = [c for c in df.columns if c not in meta_cols]

    for c in candidates:
        cl = c.lower()
        if ("tp" in cl) and ("precip" in cl):
            return c
    for c in candidates:
        cl = c.lower()
        if ("pr" in cl) and ("precip" in cl):
            return c
    for c in candidates:
        if "precip" in c.lower():
            return c

    if len(candidates) == 1:
        return candidates[0]

    raise ValueError(f"No pude identificar la columna de precipitación en {df.columns.tolist()}")

obs_simple_files = glob.glob(os.path.join(OBS_TOTAL_FOLDER, "OBS_SIMPLE__*.csv"))
obs_idw_files = glob.glob(os.path.join(OBS_TOTAL_FOLDER, "OBS_IDW__*.csv"))

obs_simple_total_dict = {}
obs_idw_total_dict = {}

for f in obs_simple_files:
    key = os.path.basename(f).replace("OBS_SIMPLE__", "").replace(".csv", "")
    df = pd.read_csv(f)
    df["date"] = normalize_date_series_to_iso(df["date"])
    obs_simple_total_dict[key] = df

for f in obs_idw_files:
    key = os.path.basename(f).replace("OBS_IDW__", "").replace(".csv", "")
    df = pd.read_csv(f)
    df["date"] = normalize_date_series_to_iso(df["date"])
    obs_idw_total_dict[key] = df

print("OBS simple cargados:", len(obs_simple_total_dict))
print("OBS IDW cargados:", len(obs_idw_total_dict))

/tmp/ipykernel_10652/239147471.py:3: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d1 = pd.to_datetime(s, errors="coerce", dayfirst=True)
/tmp/ipykernel_10652/239147471.py:3: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d1 = pd.to_datetime(s, errors="coerce", dayfirst=True)
/tmp/ipykernel_10652/239147471.py:3: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d1 = pd.to_datetime(s, errors="coerce", dayfirst=True)
/tmp/ipykernel_10652/239147471.py:3: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d1 = pd.to_datetime(s, errors="coerce", dayfirst=True)
/tmp/ipykernel_10652/239147471.py:3:

OBS simple cargados: 12
OBS IDW cargados: 12


# **Evento por escenario y percentiles de octubre**

In [3]:
scenario_files = glob.glob(os.path.join(BYSCEN_TOTAL_FOLDER, "*.csv"))
all_scenarios = sorted({
    os.path.basename(f).split("__")[1]
    for f in scenario_files
    if "__" in os.path.basename(f) and "summary" not in os.path.basename(f).lower()
})

print("Escenarios detectados:", all_scenarios)

rows = []

for fp in sorted(scenario_files):
    name = os.path.basename(fp)

    if "summary" in name.lower():
        continue

    parts = name.replace(".csv", "").split("__")
    if len(parts) < 2:
        continue

    sustax_total = parts[0]
    scenario = parts[1]

    df = pd.read_csv(fp)
    if "date" not in df.columns:
        continue

    df["date"] = normalize_date_series_to_dt(df["date"])
    df = df.dropna(subset=["date"]).copy()

    pp_col = pick_precip_col(df)
    df[pp_col] = pd.to_numeric(df[pp_col], errors="coerce")

    target_dt = pd.Timestamp(TARGET_DATE)
    event_match = df.loc[df["date"] == target_dt, pp_col]
    event_value = float(event_match.iloc[0]) if len(event_match) else np.nan

    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month

    month_2015 = df[(df["year"] == 2015) & (df["month"] == TARGET_MONTH)]
    monthly_max_2015_10 = float(month_2015[pp_col].max()) if not month_2015.empty else np.nan

    monthly_max_oct = (
        df[df["month"] == TARGET_MONTH]
        .groupby(["year", "month"])[pp_col]
        .max()
        .reset_index()
    )

    if monthly_max_oct.empty or np.isnan(monthly_max_2015_10):
        percentile = np.nan
        rank = np.nan
        n_oct = len(monthly_max_oct)
    else:
        arr = monthly_max_oct[pp_col].dropna().values
        n_oct = len(arr)
        rank = int(np.sum(arr <= monthly_max_2015_10))
        percentile = 100.0 * rank / n_oct if n_oct > 0 else np.nan

    rows.append({
        "sustax_total": sustax_total,
        "scenario": scenario,
        "event_value_mm": event_value,
        "monthly_max_2015_10_mm": monthly_max_2015_10,
        "october_percentile": percentile,
        "october_rank": rank,
        "n_october_years": n_oct
    })

df_event_scenarios = pd.DataFrame(rows).sort_values(["sustax_total", "scenario"])
display(df_event_scenarios.head(20))

df_event_scenarios.to_csv(
    os.path.join(OUT_FOLDER, "df_event_scenarios_2015.csv"),
    index=False
)

print("Tabla guardada en:")
print(os.path.join(OUT_FOLDER, "df_event_scenarios_2015.csv"))

Escenarios detectados: ['ERA5', 'SSP119', 'SSP126', 'SSP245', 'SSP370', 'SSP434', 'SSP460', 'SSP585']


,sustax_total,scenario,event_value_mm,monthly_max_2015_10_mm,october_percentile,october_rank,n_october_years
0,Sustax_ Manzanillo_LosReyesTotal,SSP119,0.343163,13.985031,73.529412,75,102
1,Sustax_ Manzanillo_LosReyesTotal,SSP126,0.892510,6.900040,65.686275,67,102
2,Sustax_ Manzanillo_LosReyesTotal,SSP245,0.275105,4.003961,27.450980,28,102
3,Sustax_ Manzanillo_LosReyesTotal,SSP370,0.712683,9.115235,63.725490,65,102
4,Sustax_ Manzanillo_LosReyesTotal,SSP434,10.515098,22.831460,47.058824,48,102
5,Sustax_ Manzanillo_LosReyesTotal,SSP460,2.296765,22.020934,34.313725,35,102
6,Sustax_ Manzanillo_LosReyesTotal,SSP585,0.387202,14.912257,88.235294,90,102
7,Sustax_CuautitlanTotal,SSP119,1.314471,12.941878,72.549020,74,102
8,Sustax_CuautitlanTotal,SSP126,1.959141,9.800521,78.431373,80,102
9,Sustax_CuautitlanTotal,SSP245,1.222880,7.989802,36.274510,37,102


Tabla guardada en:
/content/drive/MyDrive/Doctorado /Probabilidad/Precipitación/Datos Sustax/EVENT_2015_ANALYSIS/df_event_scenarios_2015.csv


# **Tabla final de validación del evento (OBS + ERA5 + SSP)**

In [4]:
rows = []

all_keys = sorted(set(obs_simple_total_dict.keys()) | set(obs_idw_total_dict.keys()))

for sustax_total in all_keys:

    df_sim = obs_simple_total_dict.get(sustax_total)
    df_idw = obs_idw_total_dict.get(sustax_total)

    v_sim = df_sim.loc[df_sim["date"] == TARGET_DATE, "pp_mm"] if df_sim is not None else pd.Series(dtype=float)
    v_idw = df_idw.loc[df_idw["date"] == TARGET_DATE, "pp_mm"] if df_idw is not None else pd.Series(dtype=float)

    obs_simple = float(v_sim.iloc[0]) if len(v_sim) else np.nan
    obs_idw = float(v_idw.iloc[0]) if len(v_idw) else np.nan

    df_sc = df_event_scenarios[df_event_scenarios["sustax_total"] == sustax_total].copy()

    if df_sc.empty:
        rows.append({
            "sustax_total": sustax_total,
            "scenario": np.nan,
            "obs_simple_mm": obs_simple,
            "obs_idw_mm": obs_idw,
            "era5_mm": np.nan,
            "era5_obs_ratio": np.nan,
            "scenario_event_mm": np.nan,
            "scenario_monthly_max_mm": np.nan,
            "scenario_oct_percentile": np.nan
        })
        continue

    era5_row = df_sc[df_sc["scenario"] == "ERA5"]
    era5_val = float(era5_row["event_value_mm"].iloc[0]) if not era5_row.empty else np.nan
    ratio = era5_val / obs_idw if (pd.notna(obs_idw) and obs_idw > 0 and pd.notna(era5_val)) else np.nan

    for _, r in df_sc.iterrows():
        rows.append({
            "sustax_total": sustax_total,
            "scenario": r["scenario"],
            "obs_simple_mm": obs_simple,
            "obs_idw_mm": obs_idw,
            "era5_mm": era5_val,
            "era5_obs_ratio": ratio,
            "scenario_event_mm": r["event_value_mm"],
            "scenario_monthly_max_mm": r["monthly_max_2015_10_mm"],
            "scenario_oct_percentile": r["october_percentile"]
        })

df_validation = pd.DataFrame(rows)

display(df_validation.head(20))

df_validation.to_csv(
    os.path.join(OUT_FOLDER, "df_event_validation_summary.csv"),
    index=False
)

print("Tabla final guardada en:")
print(os.path.join(OUT_FOLDER, "df_event_validation_summary.csv"))

,sustax_total,scenario,obs_simple_mm,obs_idw_mm,era5_mm,era5_obs_ratio,scenario_event_mm,scenario_monthly_max_mm,scenario_oct_percentile
0,Sustax_ Manzanillo_LosReyesTotal,SSP119,240.042857,201.392467,NaN,NaN,0.343163,13.985031,73.529412
1,Sustax_ Manzanillo_LosReyesTotal,SSP126,240.042857,201.392467,NaN,NaN,0.892510,6.900040,65.686275
2,Sustax_ Manzanillo_LosReyesTotal,SSP245,240.042857,201.392467,NaN,NaN,0.275105,4.003961,27.450980
3,Sustax_ Manzanillo_LosReyesTotal,SSP370,240.042857,201.392467,NaN,NaN,0.712683,9.115235,63.725490
4,Sustax_ Manzanillo_LosReyesTotal,SSP434,240.042857,201.392467,NaN,NaN,10.515098,22.831460,47.058824
5,Sustax_ Manzanillo_LosReyesTotal,SSP460,240.042857,201.392467,NaN,NaN,2.296765,22.020934,34.313725
6,Sustax_ Manzanillo_LosReyesTotal,SSP585,240.042857,201.392467,NaN,NaN,0.387202,14.912257,88.235294
7,Sustax_CuautitlanTotal,SSP119,176.400000,166.560151,NaN,NaN,1.314471,12.941878,72.549020
8,Sustax_CuautitlanTotal,SSP126,176.400000,166.560151,NaN,NaN,1.959141,9.800521,78.431373
9,Sustax_CuautitlanTotal,SSP245,176.400000,166.560151,NaN,NaN,1.222880,7.989802,36.274510


Tabla final guardada en:
/content/drive/MyDrive/Doctorado /Probabilidad/Precipitación/Datos Sustax/EVENT_2015_ANALYSIS/df_event_validation_summary.csv
